In [11]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import numpy as np
import pandas as pd 


In [12]:
df = pd.read_excel(r'..\data\Indian_Stocks_10Yrs_Prices_By_Sector.xlsx')


In [15]:
df

,Date,HDFCBANK,ICICIBANK,SBIN,LICI,BAJFINANCE,KOTAKBANK,AXISBANK,SBILIFE,HDFCLIFE,ICICIPRULI
0,2016-02-04,241.202103,173.140488,146.987061,NaN,61.404800,134.375122,373.489380,NaN,NaN,NaN
1,2016-02-05,242.592926,177.767197,151.815887,NaN,64.835640,137.504730,389.882874,NaN,NaN,NaN
2,2016-02-08,238.133041,177.045624,155.381134,NaN,64.465904,133.749191,398.665100,NaN,NaN,NaN
3,2016-02-09,235.880142,177.767197,150.777924,NaN,63.729286,134.534103,390.224365,NaN,NaN,NaN
4,2016-02-10,232.868546,175.941986,143.512070,NaN,62.708611,132.606659,384.076813,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2464,2026-01-27,926.400024,1361.400024,1053.150024,807.799988,914.700012,408.700012,1315.800049,2038.199951,720.049988,643.849976
2465,2026-01-28,932.700012,1367.699951,1063.500000,822.150024,935.150024,412.399994,1319.800049,2053.199951,728.599976,642.400024
2466,2026-01-29,935.500000,1383.599976,1066.199951,821.000000,935.150024,412.399994,1363.699951,1996.300049,727.099976,625.299988
2467,2026-01-30,929.250000,1355.000000,1077.150024,824.500000,929.849976,408.000000,1370.400024,1998.500000,NaN,636.849976


In [16]:
df = df.drop(columns=['Date'])

In [17]:
returns = df.pct_change().fillna(0).values
returns 

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.00576622,  0.02672228,  0.03285205, ...,  0.        ,
         0.        ,  0.        ],
       [-0.01838423, -0.00405909,  0.02348402, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.00300202,  0.01162537,  0.00253874, ..., -0.02771279,
        -0.00205874, -0.02661899],
       [-0.00668092, -0.0206707 ,  0.01027019, ...,  0.00110201,
         0.        ,  0.01847111],
       [-0.0076944 , -0.01535059, -0.05472776, ..., -0.01210906,
         0.        ,  0.01036357]], shape=(2469, 10))

In [18]:
X = torch.tensor(returns[-2:-1].T, dtype=torch.float)
Y = torch.tensor(returns[-1:].T, dtype=torch.float)

In [19]:
num_nodes = X.shape[0]
edge_index = torch.tensor([
    [i for i in range(num_nodes) for j in range(num_nodes) if i != j],
    [j for i in range(num_nodes) for j in range(num_nodes) if i != j]
], dtype=torch.long)

data = Data(x=X, edge_index=edge_index, y=Y)

In [21]:
edge_index

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2,
         2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5,
         5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7,
         8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9],
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 3, 4, 5, 6,
         7, 8, 9, 0, 1, 2, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 5, 6, 7, 8, 9, 0, 1, 2,
         3, 4, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 8, 9,
         0, 1, 2, 3, 4, 5, 6, 7, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8]])

In [ ]:
W1 = torch.randn(1, 16, requires_grad=True) 
W2 = torch.randn(16, 1, requires_grad=True)

optimizer = torch.optim.Adam([W1, W2], lr=0.01)

In [ ]:
adj = torch.zeros((num_nodes, num_nodes))
for i in range(edge_index.shape[1]):
    adj[edge_index[0, i], edge_index[1, i]] = 1

for epoch in range(101):
    layer1 = torch.matmul(adj, torch.matmul(data.x, W1))
    layer1 = torch.relu(layer1)
    prediction = torch.matmul(adj, torch.matmul(layer1, W2))
    
    # mse cal
    loss = torch.mean((prediction - data.y)**2)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.8f}")

Epoch 0 | Loss: 0.01259658
Epoch 20 | Loss: 0.00129540
Epoch 40 | Loss: 0.00100683
Epoch 60 | Loss: 0.00089608
Epoch 80 | Loss: 0.00078721
Epoch 100 | Loss: 0.00068995


In [24]:
print("Bank Predictions vs Reality:")
for i, name in enumerate(['HDFC', 'ICICI', 'SBI', 'LIC', 'BAJAJ', 'KOTAK', 'AXIS', 'SBILIFE', 'HDFCLIFE', 'ICICIPRU']):
    p = prediction[i].item()
    actual = data.y[i].item()
    print(f"{name}: Pred: {p:.4f} | Actual: {actual:.4f}")

Bank Predictions vs Reality:
HDFC: Pred: -0.0096 | Actual: -0.0077
ICICI: Pred: 0.0383 | Actual: -0.0154
SBI: Pred: -0.0320 | Actual: -0.0547
LIC: Pred: -0.0258 | Actual: -0.0309
BAJAJ: Pred: -0.0131 | Actual: -0.0296
KOTAK: Pred: 0.0040 | Actual: -0.0018
AXIS: Pred: -0.0264 | Actual: -0.0219
SBILIFE: Pred: -0.0225 | Actual: -0.0121
HDFCLIFE: Pred: -0.0214 | Actual: 0.0000
ICICIPRU: Pred: -0.0404 | Actual: 0.0104
